# EXP044: raw120 vs clean120
Two CPU workers; fixed five-fold Stylegram; no competition submission.


In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[key]='2'
target=pathlib.Path('/kaggle/working/exp044-payload')
assert not target.exists(), 'Repeated execution requires explicit checkpoint recovery, not Run All again'
target.mkdir()
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='806e6c76d140bbcb5bc626d646cf126d51ea2d32258e6e745b2288b224a3693d'
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
 for name in archive.namelist():
  assert pathlib.PurePosixPath(name).name==name and ':' not in name
  (target/name).write_bytes(archive.read(name))
protocol=json.loads((target/'protocol.json').read_text())
for name in ('train.csv','extra.csv','folds.csv'):
 matches=[p for p in pathlib.Path('/kaggle/input').rglob(name) if hashlib.sha256(p.read_bytes()).hexdigest()==protocol['input_hashes'][name]]
 assert len(matches)==1, ('Expected exactly one frozen input',name,len(matches))
 (target/name).write_bytes(matches[0].read_bytes())
for name,digest in protocol['input_hashes'].items(): assert hashlib.sha256((target/name).read_bytes()).hexdigest()==digest,name
root=pathlib.Path('/kaggle/working/results')
print('EXP044 frozen. CPU only; 2 workers; full 5 folds; no network. Synthetic parity gate runs before training.',flush=True)
try:
 subprocess.run([sys.executable,'-u',str(target/'guard.py'),'--payload',str(target),'--output',str(root)],check=True)
finally:
 if root.exists():
  with zipfile.ZipFile('/kaggle/working/exp044-results.zip','w',zipfile.ZIP_DEFLATED) as archive:
   for p in sorted(root.rglob('*')):
    if p.is_file(): archive.write(p,p.relative_to(root).as_posix())
